In [1]:
import torch
import torch.nn as nn

In [2]:
#============== U_NRT MODEL SETTING =================
KERNEL_SIZE = 3
PADDING = int((KERNEL_SIZE - 1) / 2)
FEATURES = [64, 128, 256, 512]

In [4]:
class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, KERNEL_SIZE, 1, PADDING, bias=False),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, KERNEL_SIZE, 1, PADDING, bias=False),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)

In [7]:
class UNET(nn.Module):

    def __init__(self, in_channels=1, out_channels=1, features=FEATURES):
        super(UNET, self).__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)

        for feature in features:
            self.downs.append(DoubleConv(in_channels, feature))
            in_channels = feature

        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        for feature in reversed(features):
            self.ups.append(
                nn.ConvTranspose3d(feature * 2,
                                   feature,
                                   kernel_size=2,
                                   stride=2))
            self.ups.append(DoubleConv(feature * 2, feature))

        self.final_conv = nn.Conv3d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)
            skip_connection = skip_connections[idx // 2]
            concat_skip = torch.cat((skip_connection, x), dim=1)
            x = self.ups[idx + 1](concat_skip)

        # return self.softmax(self.final_conv(x))   # to be noticed
        return self.final_conv(x)

In [10]:
x = torch.randn((5, 2, 64, 64, 64))
print(x.shape)


torch.Size([5, 2, 64, 64, 64])


In [11]:
model = UNET(2, 4)


In [12]:
pred = model(x)


In [13]:
print(pred.shape)


torch.Size([5, 4, 64, 64, 64])


In [14]:
print(x.shape)

torch.Size([5, 2, 64, 64, 64])
